# Unified starting-lineup model: penalty search on Colab

Runs `results/lineup_tune.py` (coarse, fine, reduced, final) on the cached design and packages the results for download.

**Runtime:** choose *Runtime > Change runtime type > High-RAM* if available. Each fold worker needs about 4 GB; the notebook sets `LINEUP_WORKERS` from the memory it finds. A GPU does not help (NumPy/SciPy CPU code).

**Inputs:** the repository branch `apm-model` and the release asset `lineup_design_cache.tar.gz` (about 250 MB) from the GitHub release `rebuild-cache-2026-09-09`. Both need a GitHub token with repo access if the repository is private.

In [ ]:
from getpass import getpass
import os, subprocess
TOKEN = getpass('GitHub token (repo read access): ')
os.environ['GH_TOKEN'] = TOKEN
!rm -rf marvel-rivals-crawler && git clone -q --branch apm-model https://{TOKEN}@github.com/PLivdan/marvel-rivals-crawler.git
%cd marvel-rivals-crawler
!git log --oneline -1

In [ ]:
# download the cached design from the GitHub release and extract it into results/
!(type gh >/dev/null 2>&1 || (curl -sSL https://github.com/cli/cli/releases/download/v2.63.2/gh_2.63.2_linux_amd64.tar.gz | tar xz && cp gh_2.63.2_linux_amd64/bin/gh /usr/local/bin/))
!gh release download rebuild-cache-2026-09-09 --repo PLivdan/marvel-rivals-crawler --pattern lineup_design_cache.tar.gz --dir results --clobber
!cd results && tar xzf lineup_design_cache.tar.gz && ls -la lineup_design | head
!python3 -c "import json; m=json.load(open('results/lineup_design/summary.json')); print('matches', m['n'], 'free', m['free'])"

In [ ]:
import os, psutil
gb = psutil.virtual_memory().total / 1e9; cpus = os.cpu_count()
workers = max(1, min(6, int(gb // 5), cpus))          # up to 6 parallel (candidate, fold) fits
os.environ['LINEUP_WORKERS'] = str(workers); os.environ['OPENBLAS_NUM_THREADS'] = str(max(1, cpus // workers)); os.environ['PYTHONPATH'] = '.'
os.environ['LINEUP_TOL'] = '1e-4'                    # looser Newton tolerance for the search; the final fit uses 1e-6
print(f'{gb:.0f} GB RAM, {cpus} CPUs -> LINEUP_WORKERS={workers}, OPENBLAS_NUM_THREADS={os.environ["OPENBLAS_NUM_THREADS"]}')

In [ ]:
# optional: resume from a partial results/lineup_tuning.json uploaded from the Mac (skip if starting fresh)
# from google.colab import files; up = files.upload(); open('results/lineup_tuning.json','wb').write(list(up.values())[0])

In [ ]:
!python3 results/lineup_tune.py coarse 0.5 2>&1 | tail -40

In [ ]:
!python3 results/lineup_tune.py fine 2>&1 | tail -30
!python3 results/lineup_tune.py reduced 2>&1 | tail -12
!python3 results/lineup_tune.py final 2>&1 | tail -6

In [ ]:
# package what the Mac needs back: the tuning log and the selected fit
!tar czf lineup_tuning_results.tar.gz results/lineup_tuning.json results/lineup_selected.npz
from google.colab import files; files.download('lineup_tuning_results.tar.gz')
# or, to keep a copy on Drive:
# from google.colab import drive; drive.mount('/content/drive'); !cp lineup_tuning_results.tar.gz /content/drive/MyDrive/

Back on the Mac, extract the archive into the repository root (it restores `results/lineup_tuning.json` and `results/lineup_selected.npz`), then run the summaries, report, bootstrap, and the locked confirmation:

```
tar xzf ~/Downloads/lineup_tuning_results.tar.gz
PYTHONPATH=. python3 results/lineup_summaries.py results/lineup_selected.npz results/lineup_selected
python3 results/make_lineup_report.py results/lineup_selected
```